In [36]:
import os
import cv2
import pickle
import shutil
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from typing import Tuple
from skimage.metrics import structural_similarity as ssim

In [37]:
# Transfer all original images into the folder

In [38]:
full_dataset_path = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset'
full_dataset_augmented_hq_path = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset_augmented_hq'

In [39]:
full_dataset_augmented_path = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset_augmented'

In [40]:

def is_cohesive_mask(mask_gray: np.ndarray,
                     thresh_method: str = 'otsu',
                     min_area: int = 50,
                     main_frac_thresh: float = 0.6,
                     max_extra_components: int = 1
                    ) -> Tuple[bool, np.ndarray]:
    """
    Decide whether a grayscale mask is coherent (one big region) or incohesive
    (many small regions), *and* return a mask image that contains only the
    dominant region(s).

    Returns
    -------
    coherent : bool
        True if one blob covers ≥ `main_frac_thresh` of all (filtered) mask pixels
        and there are at most `max_extra_components` others.
    mask_filtered : np.ndarray
        8‑bit binary image (0 or 255) where only the largest region and up to
        `max_extra_components` next‑largest regions are kept.
    """
    # ——— ensure uint8 [0,255] ———
    if mask_gray.dtype in (np.float32, np.float64):
        img = (mask_gray * 255).astype(np.uint8)
    else:
        img = mask_gray.astype(np.uint8)

    # ——— binarize ———
    if thresh_method == 'otsu':
        _, bw = cv2.threshold(img, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif thresh_method == 'adaptive':
        bw = cv2.adaptiveThreshold(img, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY,
                                   blockSize=11, C=2)
    else:  # 'fixed'
        _, bw = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    # ——— connected components ———
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bw,
                                                                  connectivity=8)
    # stats[i, cv2.CC_STAT_AREA] is area of label i
    # skip stats[0] (background)
    areas = stats[1:, cv2.CC_STAT_AREA]
    labels_list = np.arange(1, n_labels)

    # ——— filter out tiny noise blobs ———
    keep = areas >= min_area
    if not np.any(keep):
        # no component big enough
        empty_mask = np.zeros_like(bw)
        return False, empty_mask

    filtered_labels = labels_list[keep]
    filtered_areas = areas[keep]
    total_area = filtered_areas.sum()

    # ——— sort by descending area ———
    idx_sorted = np.argsort(-filtered_areas)
    sorted_labels = filtered_labels[idx_sorted]

    # ——— select top regions ———
    keep_labels = sorted_labels[: 1 + max_extra_components]

    # ——— build filtered mask ———
    mask_filtered = np.isin(labels, keep_labels).astype(np.uint8) * 255

    # ——— decision ———
    largest_area = filtered_areas[idx_sorted[0]]
    num_extras = len(filtered_labels) - 1
    coherent = (largest_area / total_area) >= main_frac_thresh and \
               num_extras <= max_extra_components

    return coherent, mask_filtered


In [16]:
os.makedirs(full_dataset_augmented_hq_path)

In [41]:
for file in os.listdir(full_dataset_path):
    path = os.path.join(full_dataset_path, file)
    # destination_path = os.path.join(full_dataset_augmented_hq_path, file)
    shutil.copy(path, full_dataset_augmented_hq_path)

In [42]:
import os
import re

def reformat_synthetic(path: str) -> str:
    """
    Auto-detects image vs. mask from filename and reformats to:
      ...-synthetic-{n}-image.png  or  ...-synthetic-{n}-mask.png
    """
    dirname, filename = os.path.split(path)
    name, ext = os.path.splitext(filename)

    # match “…_img” or “…_mask” at end of the stem
    m = re.match(r"(.+?)_(img|mask)$", name)
    if not m:
        raise ValueError(f"Filename doesn’t end with _img or _mask: {filename!r}")
    prefix, kind = m.groups()  # kind is 'img' or 'mask'
    # map 'img' → 'image'
    kind = "image" if kind == "img" else "mask"

    # prefix now e.g. "FD-031-slice-13-image_0" or similar
    # split off the index at the last underscore
    try:
        base, idx = prefix.rsplit("_", 1)
    except ValueError:
        raise ValueError(f"No index found after underscore in {prefix!r}")

    new_filename = f"{base}-synthetic-{idx}-{kind}.png"
    return os.path.join(dirname, new_filename)

In [43]:

stats = {
}

for file in tqdm(os.listdir(full_dataset_augmented_path)):
    if '_img' in file:
        cv_subject = file.split('-slice')[0]
        if cv_subject not in stats:
            stats[cv_subject] = {
                "num_total": 0,
                "num_nonzero_masks": 0,
                "num_cohesive_masks": 0,
                "num_high_ssim": 0
            }
        synthetic_image_path = os.path.join(full_dataset_augmented_path, file)
        synthetic_mask_path = synthetic_image_path.replace('_img', '_mask')
        original_image_path = None
        if '_img' in synthetic_image_path:
            # Everything at or before -image, then add .png
            original_image_path = synthetic_image_path.split('-image')[0] + '-image' + '.png'
        if original_image_path is None:
            print('fail')
            break

        # Increment total count for this subject
        stats[cv_subject]["num_total"] += 1

        original_image_mask_path = original_image_path.replace('-image', '-mask')

        original_image = cv2.imread(original_image_path)
        original_image = cv2.cvtColor(np.asarray(original_image), cv2.COLOR_RGB2GRAY)
        original_mask = cv2.imread(original_image_mask_path) 
        original_mask = cv2.cvtColor(np.asarray(original_mask), cv2.COLOR_RGB2GRAY)

        synthetic_image = cv2.imread(synthetic_image_path)
        synthetic_image = cv2.cvtColor(np.asarray(synthetic_image), cv2.COLOR_RGB2GRAY)
        synthetic_mask = cv2.imread(synthetic_mask_path)
        synthetic_mask = cv2.cvtColor(np.asarray(synthetic_mask), cv2.COLOR_RGB2GRAY)

        # Resize all images/masks to 256x256
        original_image = cv2.resize(original_image, (256, 256))
        original_mask = cv2.resize(original_mask, (256, 256))
        synthetic_image = cv2.resize(synthetic_image, (256, 256))
        synthetic_mask = cv2.resize(synthetic_mask, (256, 256))

        ssim_value = ssim(
            np.array(original_image),
            synthetic_image,
            full=True
        )[0]

        if ssim_value > 0.5:
            stats[cv_subject]["num_high_ssim"] += 1
        else:
            continue

        mask_normalized = synthetic_mask / 255.0  # Normalize to [0,1] for transparency

        has_mask = mask_normalized.flatten().sum() > 5
        if has_mask:
            stats[cv_subject]["num_nonzero_masks"] += 1

        mask_bool = synthetic_mask > 200
        percent_mask_highlighted = np.sum(mask_bool) / mask_bool.size 
        is_cohesive_mask_image, cohesive_mask = is_cohesive_mask(
            mask_normalized,
            thresh_method='otsu',
            min_area=50,
            main_frac_thresh=0.6,
            max_extra_components=1
        )
        if percent_mask_highlighted < 0.5 and is_cohesive_mask_image:
            stats[cv_subject]["num_cohesive_masks"] += 1

        # Copy the synthetic image and mask to the augmented_hq path
        synthetic_image_new_path = os.path.join(full_dataset_augmented_hq_path, os.path.basename(synthetic_image_path))
        synthetic_mask_new_path = os.path.join(full_dataset_augmented_hq_path, os.path.basename(synthetic_image_path.replace('_img', '_mask')))

        synthetic_image_new_path = reformat_synthetic(synthetic_image_new_path)
        synthetic_mask_new_path = reformat_synthetic(synthetic_mask_new_path)

        # print(synthetic_image_path, '->', synthetic_image_new_path)
        # print(synthetic_mask_path, '->', synthetic_mask_new_path)

        shutil.copy(synthetic_image_path, synthetic_image_new_path)
        shutil.copy(synthetic_mask_path, synthetic_mask_new_path)


  0%|          | 0/18720 [00:00<?, ?it/s]

100%|██████████| 18720/18720 [01:25<00:00, 218.41it/s]


In [ ]:
# print(f'Total images: {num_total}')
# print(f'High SSIM images: {num_high_ssim}')
# print(f'Non-zero masks: {num_nonzero_masks}')
# print(f'Cohesive masks: {num_cohesive_masks}')

Total images: 9000
High SSIM images: 3770
Non-zero masks: 3770
Cohesive masks: 1712


In [ ]:

synthetic_image_new_path = os.path.join(full_dataset_augmented_hq_path, os.path.basename(synthetic_image_path))
synthetic_mask_new_path = os.path.join(full_dataset_augmented_hq_path, os.path.basename(synthetic_image_path.replace('_img', '_mask')))

In [34]:
reformat_synthetic(synthetic_image_new_path)

'/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset_augmented_hq/FD-031-slice-13-image-synthetic-0-image.png'

In [35]:
reformat_synthetic(synthetic_mask_new_path)

'/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset_augmented_hq/FD-031-slice-13-image-synthetic-0-mask.png'

In [39]:
original_image_path

'/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset_augmented/FD-027-slice-35-image.png'

In [15]:
# Compare SSIM
# Compare mask quality